In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 6 13:59:45 2023

In [ ]:
@author: Praveen Singh
"""

In [ ]:
import os
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
import xarray as xr
from datetime import datetime
from pyGSI.diags import Conventional
from pathlib import Path

1. Read Dataset and convert to csv

In [ ]:
gsifile = ".diag_conv_t_ges.2021080100.nc4"
diag = Conventional(gsifile)
df = diag.get_data()
df.to_csv('./temp_gsi.csv',index=True, header=True)
print('Head of the Dataset')
print(df.head())

2. Create a new subset of few variables<br>
ewdf = df[['Station_ID', 'latitude', 'longitude', 'Pressure', 'observation']]

In [ ]:
newdf = df[['latitude', 'longitude', 'time', 'observation']]

3. Describe the data 

In [ ]:
print('Head of the sub Dataset')
print(newdf.head())
print('Tail of the sub Dataset')
print(newdf.tail())
print('Describe the sub Dataset')
print(newdf.describe())
print('Histograms of the sub Dataset')
#newdf.hist()
#plt.show()

3.1 Change time to datetime format<br>
ewdf['time'] = pd.to_datetime(newdf['time'], format='%Y-%m')<br>
ewdf['time'] = newdf['time'], format='%Y-%m-%d %H:%M:%S')<br>
rint(newdf.head())

4. Correlation 

In [ ]:
print(newdf.corr())

5. Check missing values

In [ ]:
newdf.isnull().sum()
missing_count = newdf.isnull().sum() # the count of missing values
value_count = newdf.isnull().count() # the count of all values
missing_percentage = round(missing_count / value_count * 100, 1) 
missing_df = pd.DataFrame({'count': missing_count, 'percentage': missing_percentage})
print(missing_df)

6. Create the time-series plot

In [ ]:
plt.plot(newdf['time'], newdf['observation'], linestyle='dotted')
# Add title and axis labels
plt.title('Time Series Plot')
plt.xlabel('Time')
plt.ylabel('Observations')
plt.xticks(rotation=45)
plt.xlim(2.9, 3)
#plt.show()
#exit()

7. ML/ DBSCAN: Unsupervised learning 

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

In [ ]:
print(newdf.head())
print(newdf.columns)
print(f"number of rows: {len(newdf)}")

Scale and normalize

In [ ]:
scaler = StandardScaler()

In [ ]:
newdf_s = scaler.fit_transform(newdf)
newdf_norm = pd.DataFrame(normalize(newdf_s))
print(newdf_norm.head())

In [ ]:
pca = PCA(n_components = 2)
newdf_principal = pca.fit_transform(newdf_norm)
newdf_principal = pd.DataFrame(newdf_principal)
newdf_principal.columns = ['P1', 'P2']

In [ ]:
db_model = DBSCAN(eps = 0.05, min_samples = 10).fit(newdf_principal)
labels = db_model.labels_

In [ ]:
np.unique(labels)

In [ ]:
np.histogram(labels, bins=len(np.unique(labels)))
print(np.histogram(labels, bins=len(np.unique(labels))))
#plt.hist(labels, bins=len(np.unique(labels)), log=True)
#plt.show()

In [ ]:
n_clusters = len(np.unique(labels))-1
anomaly = list(labels).count(-1)
print(f'Clusters: {n_clusters}')
print(f'Abnormal points: {anomaly}')

In [ ]:
import seaborn as sns
plt.figure()
sns.scatterplot(
    x="P1", y="P2",
    palette=sns.color_palette("hls", 10),
    data=newdf_principal,
    legend="full",
    alpha=0.3
)
#plt.show()

8. Split the dataset into training and testing parts using sklearn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import sklearn
print(sklearn.__version__)
X = newdf.iloc[:,:-1]
print(X)
y = newdf.iloc[:, 3]
print(y)

https://www.geeksforgeeks.org/how-to-split-a-dataset-into-train-and-test-sets-using-python/

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, train_size=0.7, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

8.1 Scale the training data towards unit variance (mean=0  variance=1)

In [ ]:
scaler = preprocessing.StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

8.2 Implement logistic regression model, train, and test: Supervised learning

In [ ]:
from sklearn.linear_model import LogisticRegression
reg = LogisticRegression(max_iter = 1000)
y_train = y_train.astype('int')
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
# print the train test split and the accuracy of the test
from sklearn import metrics
#print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
reg.score(X_test, y_pred)
print("Accuracy:",reg.score(X_test, y_pred))
exit()